# CSX — Mandala texture model on Colab → moving sphere texture (Unity)

Trains a StyleGAN2-ADA model on the **mandala** set on a free Colab T4, then
exports a **moving interpolation video** you can drop into Unity:
- `mandala_walk.mp4` — a seamless looping latent walk (square).
- `mandala_walk_equirect.mp4` — a 2:1 **equirectangular, horizontally-tileable**
  version that maps onto a sphere's lat-long UVs.

ADA + `--mirror` augmentation handle the small dataset; the dataset is also
pre-padded with rotated variants (mandalas are radially symmetric).

**Before you start:** upload `mandalas-256.zip` (built locally) to
`MyDrive/csx/mandalas-256.zip`. Set *Runtime → Change runtime type → T4 GPU*.

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/NVlabs/stylegan3.git
!pip install -q ninja "numpy<2" scipy click requests imageio imageio-ffmpeg pyspng
# StyleGAN3's custom ops compile on first run via ninja (~1 min).

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
CKPT = '/content/drive/MyDrive/csx/checkpoints_mandala'
os.makedirs(CKPT, exist_ok=True)
print('checkpoints ->', CKPT)

## Dataset

In [ ]:
import os, shutil
DATA_DRIVE = '/content/drive/MyDrive/csx/mandalas-256.zip'
DATA = '/content/mandalas-256.zip'
assert os.path.exists(DATA_DRIVE), f'Upload mandalas-256.zip to {DATA_DRIVE} first'
shutil.copy(DATA_DRIVE, DATA)
print('dataset ready:', DATA, os.path.getsize(DATA)//1_000_000, 'MB')

## Train
`batch=16`, `gamma=2` suit a T4 at 256px (drop to `--batch=8` on OOM). Snapshots +
`fakes*.png` land in `CKPT` on Drive. Leave it running; after a session timeout,
reconnect, re-run the GPU/Setup/Drive/Dataset cells, then **Resume**.
A usable moving texture appears well before full convergence (~few hundred kimg).

In [ ]:
cmd = (
    "python stylegan3/train.py "
    f"--outdir={CKPT} --data={DATA} "
    "--cfg=stylegan2 --gpus=1 --batch=16 --gamma=2 "
    "--mirror=1 --aug=ada --target=0.6 --snap=10 --metrics=fid50k_full"
)
print(cmd)
!{cmd}

## Resume (after a session reset)

In [ ]:
import glob, os
pkls = sorted(glob.glob(f'{CKPT}/*/network-snapshot-*.pkl'), key=os.path.getmtime)
assert pkls, 'No snapshot yet — run Train first.'
latest = pkls[-1]; print('resuming from', latest)
cmd = (
    "python stylegan3/train.py "
    f"--outdir={CKPT} --data={DATA} "
    "--cfg=stylegan2 --gpus=1 --batch=16 --gamma=2 "
    "--mirror=1 --aug=ada --target=0.6 --snap=10 --metrics=fid50k_full "
    f"--resume={latest}"
)
!{cmd}

## Export the moving texture (seamless loop)
Renders a closed-loop latent walk through several anchors and back to the start,
so the mp4 loops forever. `noise_mode='const'` makes the texture *drift* instead
of *boil*. Tune `PSI` (lower = more coherent) and `FRAMES` (longer = slower).

In [ ]:
import glob, os, pickle, numpy as np, torch, imageio.v2 as imageio
from IPython.display import HTML
from base64 import b64encode

pkls = sorted(glob.glob(f'{CKPT}/*/network-snapshot-*.pkl'), key=os.path.getmtime)
G = pickle.load(open(pkls[-1], 'rb'))['G_ema'].cuda().eval()

def slerp(a, b, t):
    a, b = a.float(), b.float()
    an, bn = a / a.norm(), b / b.norm()
    om = torch.acos((an * bn).sum().clamp(-1, 1)); so = torch.sin(om)
    if so.abs() < 1e-6:
        return (1 - t) * a + t * b
    return (torch.sin((1 - t) * om) / so) * a + (torch.sin(t * om) / so) * b

K, FRAMES, PSI = 5, 600, 0.7              # anchors, total frames, truncation
z = torch.randn(K, G.z_dim).cuda()
w = G.mapping(z, None, truncation_psi=PSI)        # [K, num_ws, w_dim]
loop = list(w) + [w[0]]                            # close the loop
seg = FRAMES // K

def render(wi):
    img = G.synthesis(wi.unsqueeze(0), noise_mode='const')
    return ((img.clamp(-1, 1) + 1) / 2 * 255)[0].permute(1, 2, 0).cpu().numpy().astype('uint8')

frames = []
for i in range(K):
    a, b = loop[i], loop[i + 1]
    for f in range(seg):
        t = f / seg
        wi = torch.stack([slerp(a[l], b[l], t) for l in range(a.shape[0])])
        frames.append(render(wi))

OUT = f'{CKPT}/mandala_walk.mp4'
imageio.mimsave(OUT, frames, fps=30)
print('saved', OUT, '-', len(frames), 'frames')
HTML(f'<video controls loop autoplay width=320 src="data:video/mp4;base64,{b64encode(open(OUT,"rb").read()).decode()}">')

## Export a sphere-ready texture (equirectangular, tileable) for Unity
Reuses the `frames` above. Each frame is cross-faded at the left/right seam so it
wraps with no vertical seam, then stretched to 2:1 (equirectangular). In Unity,
apply this video to a sphere's albedo/emission with default lat-long UVs.

Note: this is post-hoc tiling (strategy A) — quick and good for an abstract
texture; poles will pinch slightly. A fully seam-free wrap needs seam-aware
retraining (strategy B in `src/seam.py`).

In [ ]:
import numpy as np, imageio.v2 as imageio
from PIL import Image

def make_tileable(arr, blend=0.12):
    arr = arr.astype(np.float32); W = arr.shape[1]; b = max(1, int(W * blend)); o = arr.copy()
    for j in range(b):
        wgt = 0.5 * (1 - j / b); lc, rc = j, W - 1 - j
        o[:, lc] = (1 - wgt) * arr[:, lc] + wgt * arr[:, rc]
        o[:, rc] = (1 - wgt) * arr[:, rc] + wgt * arr[:, lc]
    return o.astype('uint8')

def to_equirect(arr, out_h=256):
    return np.asarray(Image.fromarray(arr).resize((2 * out_h, out_h), Image.LANCZOS))

eq = [to_equirect(make_tileable(f)) for f in frames]
OUT = f'{CKPT}/mandala_walk_equirect.mp4'
imageio.mimsave(OUT, eq, fps=30)
print('saved sphere texture (2:1 equirect):', OUT)
print('Download from Drive and use as a sphere material in Unity.')